# Day 2: RAG, step 3: store chunks + embeddings in PostgreSQL / pgvector

## Step 1: Connect to the database

In [1]:
import json
import logging
import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger("pgvector")

load_dotenv("../.env")
DB_URL = (f"postgresql://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
          f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}")

try:
    conn = psycopg.connect(DB_URL, autocommit=True)
except psycopg.OperationalError as error:
    logger.error("Cannot connect to Postgres: %s", error)
    raise RuntimeError("Database not reachable. Start it with: docker compose up -d") from None

def query_df(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[col.name for col in cur.description])

print(conn.execute("SELECT version()").fetchone()[0][:40])

PostgreSQL 17.11 (Debian 17.11-1.pgdg12+


## Step 2: Create our schema

In [2]:
schema_sql = Path("../sql/001_schema.sql").read_text(encoding="utf-8")
conn.execute(schema_sql)
register_vector(conn)

tables = conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'weather_rag' ORDER BY table_name
""").fetchall()
version = conn.execute("SELECT extversion FROM pg_extension WHERE extname = 'vector'").fetchone()[0]
print("pgvector version:", version)
print("Tables in weather_rag:", [t[0] for t in tables])

pgvector version: 0.8.7
Tables in weather_rag: ['chunks', 'documents']


## Step 3: Load the documents and chunks

In [3]:
chunks = json.loads(Path("../data/chunks.json").read_text(encoding="utf-8"))
vectors = np.load("../data/chunk_embeddings.npy")

assert len(chunks) == len(vectors), "chunks and vectors don't match - re-run the embeddings notebook"
assert vectors.shape[1] == 768, f"expected 768 numbers per vector, got {vectors.shape[1]}"
print(f"{len(chunks)} chunks, vectors {vectors.shape}")

documents = {}
for c in chunks:
    documents[c["doc_id"]] = (c["doc_id"], c["doc_title"], Path(c["source_url"]).name, c["source_url"])

with conn.transaction():
    with conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO weather_rag.documents (doc_id, title, file_name, source_url)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (doc_id) DO UPDATE
            SET title = EXCLUDED.title, file_name = EXCLUDED.file_name, source_url = EXCLUDED.source_url
        """, list(documents.values()))

        cur.executemany("""
            INSERT INTO weather_rag.chunks
                (chunk_id, doc_id, page_start, page_end, section_title, content, chars, embedding)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (chunk_id) DO UPDATE
            SET doc_id = EXCLUDED.doc_id, page_start = EXCLUDED.page_start, page_end = EXCLUDED.page_end,
                section_title = EXCLUDED.section_title, content = EXCLUDED.content,
                chars = EXCLUDED.chars, embedding = EXCLUDED.embedding
        """, [(c["chunk_id"], c["doc_id"], c["page_start"], c["page_end"], c["section_title"],
               c["text"], c["chars"], vector) for c, vector in zip(chunks, vectors)])

conn.execute("ANALYZE weather_rag.documents")
conn.execute("ANALYZE weather_rag.chunks")
print(f"Loaded {len(documents)} documents and {len(chunks)} chunks")

219 chunks, vectors (219, 768)
Loaded 4 documents and 219 chunks


## Step 4: Check what's in the database

In [4]:
query_df("""
    SELECT d.doc_id, d.title, count(*) AS chunks,
           min(c.chars) AS smallest, max(c.chars) AS largest,
           min(vector_dims(c.embedding)) AS dims
    FROM weather_rag.chunks c
    JOIN weather_rag.documents d USING (doc_id)
    GROUP BY d.doc_id, d.title
    ORDER BY d.doc_id
""")

,doc_id,title,chunks,smallest,largest,dims
0,cyclone_terms,IMD RSMC New Delhi: Cyclone terminology,27,201,599,768
1,forecast_sop,IMD: SOP for Weather Forecasting and Warning,56,102,893,768
2,heat_faq,IMD: FAQ on Heat Wave,48,189,886,768
3,monsoon_faq,IMD: FAQ on Monsoon,88,199,972,768


## Step 5: Search with SQL

In [5]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def embed_question(question):
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[question],
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768),
    )
    vector = np.array(result.embeddings[0].values, dtype="float32")
    return vector / np.linalg.norm(vector)

def db_search(question_vector, top_k=3, doc_id=None):
    sql = """
        SELECT c.chunk_id, c.doc_id, c.page_start, c.page_end, c.section_title,
               1 - (c.embedding <=> %(q)s) AS score
        FROM weather_rag.chunks c
        WHERE %(doc_id)s::text IS NULL OR c.doc_id = %(doc_id)s
        ORDER BY c.embedding <=> %(q)s
        LIMIT %(k)s
    """
    return query_df(sql, {"q": question_vector, "k": top_k, "doc_id": doc_id})

question = "What is a heat wave?"
q_vector = embed_question(question)
db_results = db_search(q_vector)
print("QUESTION:", question)
db_results

QUESTION: What is a heat wave?


,chunk_id,doc_id,page_start,page_end,section_title,score
0,heat_faq-000,heat_faq,1,1,Q. What is a heat wave?,0.739144
1,heat_faq-004,heat_faq,2,2,Q. What is a criterion for describing Heat Wav...,0.729620
2,heat_faq-005,heat_faq,2,2,Q. What are the criteria used for declaring He...,0.709940


## Step 6: Compare with the numpy search

In [6]:
numpy_scores = vectors @ q_vector
numpy_top = [chunks[i]["chunk_id"] for i in np.argsort(-numpy_scores)[:3]]

print("numpy top 3:   ", numpy_top)
print("database top 3:", db_results["chunk_id"].tolist())
print("Same results:", numpy_top == db_results["chunk_id"].tolist())

numpy top 3:    ['heat_faq-000', 'heat_faq-004', 'heat_faq-005']
database top 3: ['heat_faq-000', 'heat_faq-004', 'heat_faq-005']
Same results: True


## Step 7: `EXPLAIN`: is the HNSW index used?

In [7]:
explain_sql = """
    EXPLAIN SELECT chunk_id FROM weather_rag.chunks
    ORDER BY embedding <=> %s LIMIT 3
"""

def show_plan(title):
    print(title)
    for (line,) in conn.execute(explain_sql, (q_vector,)).fetchall():
        print("   ", line if len(line) < 110 else line[:80] + " ... (768 numbers)")

show_plan("Normal plan (219 rows):")
conn.execute("SET enable_seqscan = off")
show_plan("\nWith sequential scans turned off:")
_ = conn.execute("RESET enable_seqscan")

Normal plan (219 rows):
    Limit  (cost=48.57..48.58 rows=3 width=23)
      ->  Sort  (cost=48.57..49.12 rows=219 width=23)
            Sort Key: ((embedding <=> '[0.019373909,0.011349879,0.0020105306,-0.0754 ... (768 numbers)
            ->  Seq Scan on chunks  (cost=0.00..45.74 rows=219 width=23)

With sequential scans turned off:
    Limit  (cost=445.64..448.04 rows=3 width=23)
      ->  Index Scan using chunks_embedding_hnsw on chunks  (cost=445.64..620.38 rows=219 width=23)
            Order By: (embedding <=> '[0.019373909,0.011349879,0.0020105306,-0.07548 ... (768 numbers)


## Step 8: Search with a filter

In [8]:
question = "What is a depression?"
question_vector = embed_question(question)
print("QUESTION:", question)
print("\nAll documents:")
display(db_search(question_vector))
print("Only the cyclone glossary (doc_id = 'cyclone_terms'):")
display(db_search(question_vector, doc_id="cyclone_terms"))

QUESTION: What is a depression?

All documents:


,chunk_id,doc_id,page_start,page_end,section_title,score
0,monsoon_faq-063,monsoon_faq,4,4,12. What is the difference between monsoon dep...,0.676505
1,monsoon_faq-064,monsoon_faq,4,4,12. What is the difference between monsoon dep...,0.659332
2,monsoon_faq-065,monsoon_faq,4,4,12. What is the difference between monsoon dep...,0.658771


Only the cyclone glossary (doc_id = 'cyclone_terms'):


,chunk_id,doc_id,page_start,page_end,section_title,score
0,cyclone_terms-143,cyclone_terms,3,3,Glossary: Cyclonic storm ... Direction of move...,0.638068
1,cyclone_terms-142,cyclone_terms,3,3,Glossary: Cyclone warning ... Cyclonic disturb...,0.625183
2,cyclone_terms-141,cyclone_terms,3,3,Glossary: Cyclogenesis ... Cyclone warning bul...,0.619253
